# 🧬 Mini-Evolve

## 0. Setup

**0.1 — Fetch the exercise files.** The dataset, the frozen training loop, the seed program
and the diagrams all live in the course repo. This clones it and moves into the right folder.

In [ ]:
#@title 🔧 Fetch the exercise files { display-mode: "form" }
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE6-public"
REPO_BRANCH = "main"
HELPER     = os.path.join("06_openevolve", "exercise", "ev_viz.py")

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

_start_cwd = os.getcwd()
_repo_dir  = os.path.join(_start_cwd, REPO_NAME)   # absolute, so re-running this cell is safe

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(_repo_dir):
        print(f"Cloning the exercise repo (branch: {REPO_BRANCH})…")
        !git clone -q -b "$REPO_BRANCH" "$url" "$_repo_dir"
    else:                                 # already cloned earlier, refresh to the latest version
        print(f"Updating the exercise repo to the latest version (branch: {REPO_BRANCH})…")
        !git -C "$_repo_dir" checkout -q "$REPO_BRANCH" 2>/dev/null; \
         git -C "$_repo_dir" pull -q "$url" "$REPO_BRANCH" || echo "  (could not pull, using the existing copy)"

# Move to the REPO ROOT, the folder holding `06_openevolve/`, so imports resolve cleanly.
for _root in [_repo_dir, _start_cwd, os.path.dirname(_start_cwd),
              os.path.dirname(os.path.dirname(_start_cwd))]:
    if os.path.exists(os.path.join(_root, HELPER)):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (06_openevolve/exercise/ev_viz.py). Check your internet "
        "connection and re-run this cell.")
EXERCISE = os.path.join(os.getcwd(), "06_openevolve", "exercise")
sys.path.insert(0, EXERCISE)              # make minievolve/ and ev_viz importable
print("Working directory:", os.getcwd())


**0.2 — Install dependencies.** This adds the OpenRouter client and pins the rest.

In [ ]:
#@title 🔧 Install dependencies { display-mode: "form" }
%pip install -q -r 06_openevolve/exercise/requirements_evolve.txt

**0.3 — Import.** Four small modules. It is worth knowing what is in each before you start,
because the boundary between them *is* the lesson:

| module | what it holds | who may change it |
|---|---|---|
| `spirals` | the data, and the **frozen** training protocol | nobody, ever |
| `seed` | the starting program and the `EVOLVE-BLOCK` markers | the model, inside the markers |
| `llm` | one OpenRouter call — the mutation operator | you, if you want a different model |
| `loop` | `sample`, `mutate`, and the driver | provided |

In [ ]:
import importlib
import ev_viz
from minievolve import spirals, seed, llm, loop
for _m in (ev_viz, spirals, seed, llm, loop):
    importlib.reload(_m)

data = spirals.load_data()
CAP  = spirals.PARAM_CAP
print(f"train {tuple(data.X_train.shape)} · val {tuple(data.X_val.shape)} · "
      f"test {tuple(data.X_test.shape)}")
print(f"parameter cap: {CAP} · training budget: {spirals.STEPS} steps (frozen)")

**0.4 — Check the OpenRouter key.** If this fails, you can still do the entire notebook: the run later in the notebook falls back to a recorded one, but fixing the key now is nicer than discovering the problem in twenty minutes.

In [ ]:
try:
    print(llm.check_api_key())
    LIVE = True
except Exception as e:
    print(f"⚠️  No working key ({type(e).__name__}: {str(e)[:120]})")
    print("    That's survivable: Part 3 will replay a recorded run instead.")
    LIVE = False

## 1. The problem

Imagine you need to solve a classification problem where the data forms spirals. Each point has two coordinates, x and y, and belongs to one of three classes. The classes wind around each other, so the boundary between them is curved and a straight line cannot separate them. The metric is accuracy on points the model has not seen during training.

In [ ]:
#@title 📊 The data { display-mode: "form" }
ev_viz.problem_and_search(data)

In this notebook we do not train or fine tune the model by hand. Instead we use an evolve system that writes and improves the model for us. A language model proposes new versions of the code, and a scoring function that we write decides which versions are kept.

Here is the starting program, and what the search is and is not allowed to change.

In [ ]:
#@title 📄 The seed program { display-mode: "form" }
print(seed.SEED_PROGRAM)

In [ ]:
#@title 🔒 The evolve block: what the model may rewrite { display-mode: "form" }
ev_viz.frozen_vs_evolvable()

The model always takes an input of shape (N, 2), the two coordinates, and must return an output of shape (N, 3), one score per class.

In [ ]:
#@title 🧠 Quick check { display-mode: "form" }
ev_viz.mc_quiz("output_shape")

In [ ]:
#@title 🧠 Quick check { display-mode: "form" }
ev_viz.mc_quiz("frozen")

This is generation 0: the starting point, before any evolution happens.

In [ ]:
#@title 📈 Train the seed { display-mode: "form" }
program = seed.run_program(seed.SEED_PROGRAM)
baseline = spirals.train_and_evaluate(program, data)

print(f"parameters       {baseline.n_params}")
print(f"train accuracy   {baseline.train_accuracy:.1%}")
print(f"val accuracy     {baseline.val_accuracy:.1%}")
print(f"training time    {baseline.seconds:.2f}s for both seeds")

ev_viz.plot_boundary(baseline.models[0], data, "seed program: nn.Linear(2, 3)", split="val")

## 2. How it works

How does it do it? It repeats a loop of four steps: sample, mutate, evaluate, store. One full pass through the four steps is one generation.

In [ ]:
#@title 🗺️ The pipeline { display-mode: "form" }
ev_viz.pipeline_diagram()

Note: we are building a small version of OpenEvolve, which we call mini-evolve.

Next we go through the four steps one by one. You will implement all four, plus the loop that ties them together.

## 3. Sample (Task 1)

This is the step where we select the parent, the program that will be rewritten next. Our first parent is the plain linear model. The parent pool is divided into cells based on the number of parameters: tiny, small, medium, and large. Each cell keeps its own best program. We pick the parent with a simple rule: two thirds of the time we take the best program of a random cell, and one third of the time we take the overall best program.

In [ ]:
#@title 🗂️ The parent pool { display-mode: "form" }
ev_viz.sample_rule_diagram()

**🎯 Task 1.** Write `sample`: pick the parent using the 2/3 explore, 1/3 exploit rule, then pick its rivals, the two best plus one more at random.

In [ ]:
import random

def sample(archive: dict, rng: random.Random) -> tuple[loop.Candidate, list[loop.Candidate]]:
    """Pick a parent to mutate, and a few rivals to show alongside it.

    Two thirds of the time it picks the champion of a random cell. One third of the time
    it picks the single overall best program. Then it gathers a few rivals: the two best
    remaining programs, plus one more picked at random from what is left, so the prompt
    shows a landscape and not just a single example.
    """
    residents = list(archive.values())                     # every cell's current champion
    if rng.random() < 0.34:
        #🎯TODO: 1/3 of the time, exploit: parent = the single overall best program,
        # the highest-scoring candidate across every cell. Use max(residents, key=...).
        parent = None
    else:
        #🎯TODO: 2/3 of the time, explore: parent = the champion of a random cell,
        # picked uniformly at random from `residents` (use rng.choice).
        parent = None

    rivals = [c for c in residents if c is not parent]      # everyone else, as context
    rivals.sort(key=lambda c: c.score, reverse=True)
    #🎯TODO: top = the two best remaining rivals. `rivals` is already sorted best
    # first, so this is a slice.
    top = None
    #🎯TODO: diverse = one more rival picked at random from what's left (rivals[2:])
    # via rng.sample, but only if there is anything left to pick from (else an empty list).
    diverse = None
    return parent, top + diverse                            # deliberately not all top

This is how we check your implementation is correct.

In [ ]:
#@title ✅ Check { display-mode: "form" }
_test_archive = {i: loop.Candidate(code=seed.SEED_PROGRAM, score=s, index=i)
                 for i, s in enumerate([0.30, 0.42, 0.55, 0.68, 0.81])}

# same seed, same draw as the reference implementation, across several seeds
for _s in range(12):
    _p1, _r1 = sample(_test_archive, random.Random(_s))
    _p2, _r2 = loop.sample(_test_archive, random.Random(_s))
    assert _p1.index == _p2.index, f"seed {_s}: parent does not match loop.sample"
    assert [c.index for c in _r1] == [c.index for c in _r2], \
        f"seed {_s}: rivals do not match loop.sample"

# statistical check: parent is the overall best about 0.34 + 0.66/5 of draws
_rng = random.Random(0)
_hits = sum(sample(_test_archive, _rng)[0].score == 0.81 for _ in range(6000))
_expected = 0.34 + 0.66 / 5
_got = _hits / 6000
assert abs(_got - _expected) < 0.03, \
    f"parent-selection ratio looks off: got {_got:.3f}, expected about {_expected:.3f}"

print(f"parent is the overall best about {_got:.1%} of draws (expected about {_expected:.1%}) \u2705")
print("Matches loop.sample across 12 seeds \u2705")

## 4. Mutate (Task 2)

This is the step where we call the language model. The prompt has two parts. The system prompt is fixed and explains the problem, the rules, and the reply format. The user prompt changes every generation and contains the parent program, a few rival programs, and any rejected attempts with the reason they were rejected. The model returns a new model definition and a new optimizer.

In [ ]:
PROBLEM = """You are optimising a small PyTorch program by rewriting one block of it.

The program classifies points from a noisy three-class 2D spiral. It must define:
  build_model()          -> nn.Module taking (batch, 2) and returning (batch, 3) logits
  build_optimizer(model) -> a torch optimizer"""

HARD_RULES = """Hard rules, enforced by the scorer:
  - AT MOST {cap} TRAINABLE PARAMETERS. This is checked exactly and a program over the
    limit scores zero, however accurate it is. Count before you answer:
    nn.Linear(a, b) has a*b + b parameters. So 2->16->16->3 is
    (2*16+16) + (16*16+16) + (16*3+3) = 48 + 272 + 51 = 371. A single 64-wide hidden
    layer is already 4547 and is REJECTED. Stay well under the limit.
  - only torch and torch.nn; no file, network or data access
  - the training loop, the number of gradient steps, the loss and the data are FIXED
    outside your block. You cannot change how long it trains."""

ARCHITECTURE = """You may change architecture, depth, width, activations, initialisation, the optimizer and
its hyperparameters. You may define nn.Module subclasses inside the block."""

REPLY = """Reply with the replacement block ONLY, inside a single ```python fence. Do not repeat the
import lines. No commentary."""

print("Four parts loaded: PROBLEM, HARD_RULES, ARCHITECTURE, REPLY")

**🎯 Task 2.** Build the system prompt by joining the four parts, in order, separated by blank lines.

In [ ]:
#🎯TODO: join PROBLEM, HARD_RULES, ARCHITECTURE and REPLY, in that order, into one
# string, with a blank line between each part ("\n\n".join([...])).
SYSTEM_PROMPT = None
loop.SYSTEM_PROMPT = SYSTEM_PROMPT                                        # wire it into the loop

This is how we check your implementation is correct.

In [ ]:
#@title ✅ Check { display-mode: "form" }
_original = (
    "You are optimising a small PyTorch program by rewriting one block of it.\n\n"
    "The program classifies points from a noisy three-class 2D spiral. It must define:\n"
    "  build_model()          -> nn.Module taking (batch, 2) and returning (batch, 3) logits\n"
    "  build_optimizer(model) -> a torch optimizer\n\n"
    "Hard rules, enforced by the scorer:\n"
    "  - AT MOST {cap} TRAINABLE PARAMETERS. This is checked exactly and a program over the\n"
    "    limit scores zero, however accurate it is. Count before you answer:\n"
    "    nn.Linear(a, b) has a*b + b parameters. So 2->16->16->3 is\n"
    "    (2*16+16) + (16*16+16) + (16*3+3) = 48 + 272 + 51 = 371. A single 64-wide hidden\n"
    "    layer is already 4547 and is REJECTED. Stay well under the limit.\n"
    "  - only torch and torch.nn; no file, network or data access\n"
    "  - the training loop, the number of gradient steps, the loss and the data are FIXED\n"
    "    outside your block. You cannot change how long it trains.\n\n"
    "You may change architecture, depth, width, activations, initialisation, the optimizer and\n"
    "its hyperparameters. You may define nn.Module subclasses inside the block.\n\n"
    "Reply with the replacement block ONLY, inside a single ```python fence. Do not repeat the\n"
    "import lines. No commentary."
)
assert SYSTEM_PROMPT == _original, "the four parts must join back into exactly the original prompt"
print(f"SYSTEM_PROMPT wired into the loop \u2705  ({len(SYSTEM_PROMPT)} characters)")

This is the user message MUTATE sends alongside your `SYSTEM_PROMPT`: the parent program, scored rivals, and, once there are failures, the rejected attempts and why they were rejected.

**🎯 Task 2 (continued).** Print the user prompt MUTATE actually sends, using `loop.build_prompt`.

In [ ]:
_parent = loop.Candidate(code=seed.SEED_PROGRAM, score=0.447,
                         metrics={"params": 9, "accuracy": 0.448})
_rival  = loop.Candidate(code=seed.splice_block(seed.SEED_PROGRAM,
    "def build_model():\n    return nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 3))\n\n\n"
    "def build_optimizer(model):\n    return torch.optim.Adam(model.parameters(), lr=0.05)\n"),
    score=0.61, metrics={"params": 51, "accuracy": 0.615})

#🎯TODO: call loop.build_prompt with _parent, [_rival] and CAP, then print the
# result, to see the exact user message MUTATE sends.
raise NotImplementedError("finish this cell: print the assembled user prompt")

## 5. Evaluate (Task 3)

This step receives the candidate code as a string and returns a dictionary with the score, the accuracy, and the number of parameters. You implement the rules: a program over the parameter cap scores zero, and a program that fails to run or produces NaN scores zero. Otherwise the score is the validation accuracy minus a small size penalty.

**🎯 Task 3.** Write the fitness function.

In [ ]:
def evaluate(code: str) -> dict:
    try:
        program = seed.run_program(code)               # run the candidate under the frozen protocol
        result  = spirals.train_and_evaluate(program, data)
    except Exception as e:
        return {"score": 0.0, "note": f"{type(e).__name__}: {e}"}

    #🎯TODO: did the candidate fail to run at all (crash, NaN loss, wrong output
    # shape)? `result.ok` is already False for exactly those cases.
    if None:
        return {"score": 0.0, "note": result.error}
    #🎯TODO: did it break the parameter cap? Compare result.n_params to CAP.
    if None:
        return {"score": 0.0, "note": f"{result.n_params} params over the {CAP} cap"}

    #🎯TODO: score = validation accuracy, minus a small penalty proportional to
    # size (keep the penalty small, around 0.05 at the cap, so it only decides close ties).
    score = None
    return {"score": score, "accuracy": result.val_accuracy, "params": result.n_params}

This is how we check your implementation is correct.

In [ ]:
#@title ✅ Check { display-mode: "form" }
_prog = lambda block: seed.splice_block(seed.SEED_PROGRAM, block)
_seed_score = evaluate(seed.SEED_PROGRAM)
_big = _prog("def build_model():\n    return nn.Sequential(nn.Linear(2, 64), nn.ReLU(), "
             "nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))\n\n\n"
             "def build_optimizer(model):\n    return torch.optim.Adam(model.parameters(), lr=0.05)\n")
_wrong = _prog("def build_model():\n    return nn.Linear(2, 7)\n\n\n"
               "def build_optimizer(model):\n    return torch.optim.Adam(model.parameters(), lr=0.05)\n")

print(f"seed program : {_seed_score['score']:.4f}  ({_seed_score.get('params')} params)")
print(f"4547 params  : {evaluate(_big)['score']:.4f}  <- must be exactly 0")
print(f"wrong shape  : {evaluate(_wrong)['score']:.4f}  <- must be exactly 0")

assert 0.40 < _seed_score["score"] < 0.46,  "the seed should score just under its 45% accuracy"
assert evaluate(_big)["score"] == 0.0,      "over the cap must score zero, whatever its accuracy"
assert evaluate(_wrong)["score"] == 0.0,    "wrong output shape must score zero"
assert abs(evaluate(seed.SEED_PROGRAM)["score"] - _seed_score["score"]) < 1e-9, \
    "evaluation must be repeatable, a noisy score selects noise"
print("\nA scoreboard the search cannot argue with \u2705")

## 6. Store (Task 4)

This step decides whether the new candidate is kept. Find the cell for its parameter count. Keep it if the cell is empty or if it beats the current best of that cell. Do not compare it to programs in other cells.

**🎯 Task 4.** Write the archive rule.

In [ ]:
def store(archive: dict, cand) -> dict:
    #🎯TODO: cell = which size class this candidate is in (loop.param_bucket on
    # its param count, cand.metrics.get("params", 0)).
    cell     = None
    champion = archive.get(cell)                                 # None if the cell is empty
    #🎯TODO: the newcomer takes the slot if the cell is empty (champion is None),
    # or if it beats the current champion's score. Do not compare it to other cells.
    if None:
        archive[cell] = cand
    return archive

This is how we check your implementation is correct.

In [ ]:
#@title ✅ Check { display-mode: "form" }
_mk = lambda score, params, gen=0: loop.Candidate(
    code=seed.SEED_PROGRAM, score=score, generation=gen, metrics={"params": params})

_a = {}
for _c in [_mk(0.79, 480), _mk(0.74, 70), _mk(0.62, 75), _mk(0.81, 470)]:
    _a = store(_a, _c)

print({loop.BUCKET_LABELS[k]: round(v.score, 2) for k, v in sorted(_a.items())})

assert len(_a) == 2, "two size classes were used, so two cells should be occupied"
assert any(c.metrics["params"] == 70 for c in _a.values()), \
    "the small 0.74 program must survive the big 0.79 one. That is the entire point"
assert max(c.score for c in _a.values()) == 0.81, "the better big model should displace the worse"
assert all(c.score != 0.62 for c in _a.values()), "0.62 must not displace 0.74 in the same cell"
print("\nA small program that a greedy archive would have deleted \u2705")

In [ ]:
#@title 🧠 Quick check { display-mode: "form" }
ev_viz.mc_quiz("archive")

## 7. The evolve loop (Task 5)

This step ties the other three together: sample a parent, mutate it, evaluate the result, and store it if it earns a place. Repeat for a fixed number of generations.

**🎯 Task 5.** Write a simplified `evolve` loop from your own `sample`, `mutate`, `evaluate` and `store`.

In [ ]:
def evolve_simple(sample_fn, mutate_fn, evaluate_fn, store_fn, generations, cap, rng_seed=0):
    """A simplified loop.evolve: sample, mutate, evaluate, store, repeat."""
    first = loop.Candidate(code=seed.SEED_PROGRAM, generation=0, index=0)  # generation 0
    result = evaluate_fn(first.code)
    first.score = result.get("score", 0.0)
    first.metrics = {k: v for k, v in result.items() if k != "score"}

    archive = store_fn({}, first)
    history = [first]
    rng = random.Random(rng_seed)

    for gen in range(1, generations + 1):
        #🎯TODO SAMPLE: call sample_fn with (archive, rng); it returns a
        # (parent, inspirations) tuple, exactly like loop.sample from Task 1.
        parent, inspirations = None, None
        #🎯TODO MUTATE: call mutate_fn with (parent, inspirations, cap) to get the
        # new candidate's source code.
        code_ = None
        child = loop.Candidate(code=code_, generation=gen, parent=parent.index, index=len(history))
        result = evaluate_fn(child.code)                     # EVALUATE
        child.score = result.get("score", 0.0)
        child.metrics = {k: v for k, v in result.items() if k != "score"}
        #🎯TODO STORE: call store_fn with (archive, child); it returns the
        # (possibly updated) archive, exactly like your store from Task 4.
        archive = None
        history.append(child)

    return {"archive": archive, "history": history, "best": max(history, key=lambda c: c.score)}

This is how we check your implementation is correct.

In [ ]:
#@title ✅ Check { display-mode: "form" }
def _mock_mutate(parent, inspirations, cap):
    # a deterministic stand-in for MUTATE, so this check needs no API key or network
    return seed.splice_block(parent.code,
        "def build_model():\n    return nn.Linear(2, 3)\n\n\n"
        "def build_optimizer(model):\n    return torch.optim.SGD(model.parameters(), lr=0.05)\n")

_run_a = evolve_simple(sample, _mock_mutate, evaluate, store, generations=3, cap=CAP, rng_seed=7)
_run_b = evolve_simple(sample, _mock_mutate, evaluate, store, generations=3, cap=CAP, rng_seed=7)

assert len(_run_a["history"]) == 4, "history should hold the seed plus one candidate per generation"
assert len(_run_b["history"]) == 4
assert [c.score for c in _run_a["history"]] == [c.score for c in _run_b["history"]], \
    "the same seed must reproduce the same run"
assert len(_run_a["archive"]) >= 1, "the archive should hold at least one cell"
print(f"ran {len(_run_a['history']) - 1} generations deterministically, "
      f"archive holds {len(_run_a['archive'])} cell(s) \u2705")

## 8. Run the loop

Now that every piece is in place, we run the loop for a fixed number of generations and watch it work.

This runs your own `evolve_simple` from Task 5, for real: the same function, not a frozen stand-in. With no API key, MUTATE is replaced by a stand-in that plays back the recorded run's mutations in order, so `evaluate` and `store` are still exercised on real evolved code and the scores reproduce exactly.

In [ ]:
GENERATIONS = 15
on_step = ev_viz.run_tracker()

if LIVE:
    print("Running live, one API call per generation.\n")
    def _real_mutate(parent, inspirations, cap):
        return loop.mutate(parent, inspirations, cap)
    run = evolve_simple(sample, _real_mutate, evaluate, store,
                        generations=GENERATIONS, cap=CAP, rng_seed=0)
else:
    print("No API key: MUTATE is replaced by a stand-in that replays the recorded run's "
          "mutations, in order, through YOUR evolve_simple.\n")
    _recorded = iter(loop.load_run("recorded_run")[1:])   # [0] is the seed, not a mutation
    def _replay_mutate(parent, inspirations, cap):
        return next(_recorded).code
    run = evolve_simple(sample, _replay_mutate, evaluate, store,
                        generations=GENERATIONS, cap=CAP, rng_seed=0)

for _c in run["history"]:
    on_step(_c, run["archive"])

history, archive, best = run["history"], run["archive"], run["best"]
print(f"\nbest score {best.score:.4f} \u00b7 {best.metrics.get('params')} params \u00b7 "
      f"val accuracy {best.metrics.get('accuracy', float('nan')):.1%}")

In [ ]:
#@title 📈 Score and size over generations { display-mode: "form" }
ev_viz.plot_progress(history)

## 9. The archive across sizes

Each cell of the archive keeps the best program of its size class. Reading the archive from tiny to large shows how accuracy changes as the parameter budget grows. The parameters do not grow inside one program. Each point is a different program, one per size cell.

In [ ]:
#@title 🗄️ The archive, cell by cell { display-mode: "form" }
ev_viz.archive_cards(archive)

## 10. Champion programs and the final boundary

Here are the champion programs and the boundary the best one draws. The seed drew straight wedges. The evolved program draws a curved boundary that separates the three spirals.

In [ ]:
#@title 📄 Champion block { display-mode: "form" }
print(best.block)

In [ ]:
#@title 🌀 Boundary comparison { display-mode: "form" }
best_result = spirals.train_and_evaluate(seed.run_program(best.code), data)
ev_viz.plot_boundary([baseline.models[0], best_result.models[0]], data,
                     ["seed: nn.Linear(2, 3)", f"evolved: {best.metrics.get('params')} params"],
                     split="val")

## 11. The held-out test split

In [ ]:
rows = [("seed program", baseline, spirals.test_accuracy(baseline, data)),
        ("evolved best", best_result, spirals.test_accuracy(best_result, data))]

print(f"{'':<15}{'params':>8}{'train':>9}{'val':>9}{'test':>9}{'val−test':>10}")
for name, r, te in rows:
    print(f"{name:<15}{r.n_params:>8}{r.train_accuracy:>9.1%}{r.val_accuracy:>9.1%}"
          f"{te:>9.1%}{r.val_accuracy - te:>+10.1%}")

In [ ]:
#@title 🧠 Quick check { display-mode: "form" }
ev_viz.mc_quiz("validation")

In [ ]:
#@title 🧠 Reflection { display-mode: "form" }
ev_viz.mc_quiz("test_gap")

## 12. Store versus greedy

Same candidates, same scores. Only the retention rule changes.

In [ ]:
def store_greedy(archive, cand):
    """The instinctive rule: remember the best, forget everything else."""
    champ = archive.get("best")
    if champ is None or cand.score > champ.score:
        archive["best"] = cand
    return archive

greedy = loop.replay(history, store_greedy)
greedy_archive = greedy["archive"]

print(f"quality-diversity : {len(archive)} programs kept, "
      f"{sorted(c.metrics.get('params', 0) for c in archive.values())} params")
print(f"greedy            : {len(greedy_archive)} program kept, "
      f"{greedy['best'].metrics.get('params')} params")

# Which mutations were bred from a parent a greedy archive would already have deleted?
orphaned = []
for c in history:
    if c.parent is None:
        continue
    champion_then = max((x for x in history if x.index < c.index), key=lambda x: x.score)
    if c.parent != champion_then.index:
        orphaned.append(c)

print(f"\n{len(orphaned)} of {len(history) - 1} mutations were bred from a parent that was "
      f"NOT the\nrunning champion. Under the greedy rule, none of them could have happened.")

if best.index in {c.index for c in orphaned}:
    print(f"\n\u26a0\ufe0f  Including the winner. The best program (generation {best.generation}, "
          f"score {best.score:.4f})\n    descends from candidate #{best.parent}, which a "
          f"greedy archive would have thrown away.")

ev_viz.plot_archive_vs_greedy(archive, greedy_archive, history)

Look at the graph and the counts above it. The quality-diversity archive keeps one champion per size cell, so a small early program can sit untouched for many generations even while larger programs score higher elsewhere. The greedy archive keeps only one program at a time: whatever currently scores best. Every time a new best appears, greedy deletes whatever it was holding, including every other program that generation produced.

A candidate is bred from a parent when it is the result of mutating that parent's code. Under quality-diversity, a candidate that is not the current best can still survive in its own cell and later be sampled as a parent again. Under greedy, only the current best is ever available as a parent, so any lineage that falls behind is deleted and can never be bred from again, even if it would have eventually led to the winner. Both archives end up holding the same final champion, because the champion is the champion regardless of what else survived alongside it. The cost greedy pays is invisible until you ask what got thrown away on the way there.

In [ ]:
#@title 🧠 Reflection { display-mode: "form" }
ev_viz.mc_quiz("greedy_vs_qd")

## 13. Congratulations 🎉

You built mini-evolve: sample, mutate, evaluate, store, and the loop that ties them together, all five pieces working. You wrote the parent-selection rule, the scoring function, the archive, and the loop itself. That is the whole method, not a simplified stand-in for it.

Take the seed program's nine parameters and the evolved program's curved boundary as a small proof: a system with no understanding of spirals, guided only by the score you wrote, found something better than where it started. Nice work! 🚀